In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import year, col, sum
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, BooleanType
from functools import reduce

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/zulu-17.jdk/Contents/Home" # ignore this using your java setting




<h1>
DATA OVERIVIEW
</h1>

<text>The Following dataset is a subset data from "NOAA Global Surface Summary of Day". It covers year from 2022-2025, 4 years in total, and includes data for 50 unique weather station. 

The following are data elements included in the dataset (as available from each station), and how they are denoted in dataframe

* Mean temperature (.1 Fahrenheit). TEMP
* Mean dew point (.1 Fahrenheit). DEWP
* Mean sea level pressure (.1 mb). SLP
* Mean station pressure (.1 mb). STP
* Mean visibility (.1 miles). VISIB
* Mean wind speed (.1 knots), WDSP
* Maximum sustained wind speed (.1 knots). MXSPD
* Maximum wind gust (.1 knots). GUST
* Maximum temperature (.1 Fahrenheit). MAX
* Minimum temperature (.1 Fahrenheit). MIN
* Precipitation amount (.01 inches). PRCP
* Snow depth (.1 inches). SNDP
* Indicator for occurrence of: Fog. Rain or Drizzle. Snow or Ice Pellets. Hail. Thunder. Tornado/Funnel Cloud. FRSHTT
</text>

<h1>
I) DATA EXTRACTION
<h1>

In [2]:
""" 
The following code will read the data, year 2022-2025, from azure blob and turn them into spark Dataframe

- Running this code will automatically overwrite the data into raw data
"""


spark = SparkSession.builder.appName("NOAA Pipeline").getOrCreate()
storage_account_name = "noaapipiline"
container_name = "rawdata"

# replace sas token, expired on April 20th
load_dotenv()
sas_token = os.environ.get("SAS_TOKEN") 
if not sas_token:
    raise ValueError("No valid SAS token")

spark.conf.set(
    f"fs.azure.sas.{container_name}.{storage_account_name}.blob.core.windows.net", sas_token
)

# data schema define 
myschema = StructType([
    StructField("STATION", IntegerType(), False),
    StructField("DATE", DateType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
    StructField("ELEVATION", DoubleType(), True),
    StructField("NAME", StringType(), True),
    StructField("TEMP", DoubleType(), True),
    StructField("TEMP_ATTRIBUTES", IntegerType(), True),
    StructField("DEWP", DoubleType(), True),
    StructField("DEWP_ATTRIBUTES", IntegerType(), True),
    StructField("SLP", DoubleType(), True),
    StructField("SLP_ATTRIBUTES", IntegerType(), True),
    StructField("STP", DoubleType(), True),
    StructField("STP_ATTRIBUTES", IntegerType(), True),
    StructField("VISIB", DoubleType(), True),
    StructField("VISIB_ATTRIBUTES", IntegerType(), True),
    StructField("WDSP", DoubleType(), True),
    StructField("WDSP_ATTRIBUTES", IntegerType(), True),
    StructField("MXSPD", DoubleType(), True),
    StructField("GUST", DoubleType(), True),
    StructField("MAX", DoubleType(), True),
    StructField("MAX_ATTRIBUTES", StringType(), True),
    StructField("MIN", DoubleType(), True),
    StructField("MIN_ATTRIBUTES", StringType(), True),
    StructField("PRCP", DoubleType(), True),
    StructField("PRCP_ATTRIBUTES", StringType(), True),
    StructField("SNDP", DoubleType(), True),
    StructField("FRSHTT", StringType(), True),
])

# read file from azureblob and return as spark dataframe
def read_csv(year:int) -> DataFrame:
    print(f"Extracting files from azure blob for {year}")
    file_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{year}"

    df = (spark.read
      .format("csv")
      .option("header", "true")
      .schema(myschema)
      .load(file_path))

    print("Successful extracting the file")
    return df

df_2022 = read_csv(2022)
df_2023 = read_csv(2023)
df_2024 = read_csv(2024)
df_2025 = read_csv(2025)

print("=" * 20)
print("Finishing extracting all files for all years")
print("=" * 20)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 10:39:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Extracting files from azure blob for 2022
Successful extracting the file
Extracting files from azure blob for 2023
Successful extracting the file
Extracting files from azure blob for 2024
Successful extracting the file
Extracting files from azure blob for 2025
Successful extracting the file
Finishing extracting all files for all years


<h1>
II) DATA_EXPLORATION
<h1>

In [3]:
# combine all year dataframe into one
df_list = [df_2022,df_2023, df_2024, df_2025]
df_combine = reduce(DataFrame.union, df_list)

In [ ]:
"""
The following code is a data exploration for year from 2022-2025
By running the code, it will process the following task

- coutning total number of records
- counting numbers of unique stations
- counting number of years that data covers
- finding all record that have null values in TEMP columns, mean temperature
- finding all records that have mean temperature bigger than max temperature, TEMP > MAX
- finding all records that have mean temperature smaller than min temperature, TEMP < MIN
"""



# counting total records
number_record = df_combine.count()
number_columns = len(df_combine.columns)

print("=" * 20)
print(f"Dataset size:")
print(f"number of records:{number_record}")
print(f"number of columns:{number_columns} ")
print("=" * 20)

# counting unique stations
stations = df_combine.select("STATION").distinct().count()

print("=" * 20)
print(f"number of unique weather station")
print(stations)
print("=" * 20)

# findings length of year covers
df_year = df_combine.withColumn("YEAR", year("DATE"))
years_cover = df_year.select("YEAR").distinct().count()

print("=" * 20)
print(f"number of year cover")
print(years_cover)
print("=" * 20)


#check null values in everycolumn
df_null= df_combine.select(
    [sum((col(c).isNull()| (col(c) == 9999.9))
         if isinstance(df_combine.schema[c].dataType, DoubleType)
         else col(c).isNull()
         .cast("int"))
         .alias(c) 
         for c in df_combine.columns])

print("="*20)
print(f"number of null for each columns")
df_null.show()
print("="*20)

# check null and invalid values for temperature field
df_check = (
    df_combine.filter((col("TEMP").isNull()) | (col("TEMP") < col("MIN")) | (col("TEMP") > col("MAX")))
)
invalid_temp_records = df_check.count()

print("="*20)
print(f"sample of invalid data in temperature field")
df_check.select("TEMP", "MAX", "MIN").show(10)
print(f"Number of records have null and invalid temperature:")
print(invalid_temp_records)
print("="*20)




Dataset size:
number of records:48404
number of columns:28 


number of unique weather station
50


number of year cover
4


{"ts": "2026-04-08 10:56:33.331", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve \"((STATION IS NULL) OR STATION)\" due to data type mismatch: the left and right operands of the binary operator have incompatible types (\"BOOLEAN\" and \"INT\"). SQLSTATE: 42K09", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "or", "errorClass": "DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o59.select.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve \"((STATION IS NULL) OR STATION)\" due to data type mismatch: the left and right operands of the binary operator have incompatible types (\"BOOLEAN\" and \"INT\"). SQLSTATE: 42K09;\n'Project ['sum(cast('`=`((isnull(STATION#0) OR STATION#0), 9999.9) as int)) AS STATION#547, 'sum(ca

AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve "((STATION IS NULL) OR STATION)" due to data type mismatch: the left and right operands of the binary operator have incompatible types ("BOOLEAN" and "INT"). SQLSTATE: 42K09;
'Project ['sum(cast('`=`((isnull(STATION#0) OR STATION#0), 9999.9) as int)) AS STATION#547, 'sum(cast('`=`((isnull(DATE#1) OR DATE#1), 9999.9) as int)) AS DATE#548, 'sum(cast('`=`((isnull(LATITUDE#2) OR LATITUDE#2), 9999.9) as int)) AS LATITUDE#549, 'sum(cast('`=`((isnull(LONGITUDE#3) OR LONGITUDE#3), 9999.9) as int)) AS LONGITUDE#550, 'sum(cast('`=`((isnull(ELEVATION#4) OR ELEVATION#4), 9999.9) as int)) AS ELEVATION#551, 'sum(cast(((isnull(NAME#5) OR cast(NAME#5 as boolean)) = 9999.9) as int)) AS NAME#552, 'sum(cast('`=`((isnull(TEMP#6) OR TEMP#6), 9999.9) as int)) AS TEMP#553, 'sum(cast('`=`((isnull(TEMP_ATTRIBUTES#7) OR TEMP_ATTRIBUTES#7), 9999.9) as int)) AS TEMP_ATTRIBUTES#554, 'sum(cast('`=`((isnull(DEWP#8) OR DEWP#8), 9999.9) as int)

In [ ]:
df_null = df_combine.select([
    sum(
        (col(c).isNull() | (col(c) == 9999.9) | (col(c) == 99.9) | (col(c) == 999.9)).cast("int")
        if isinstance(df_combine.schema[c].dataType, DoubleType)
        else col(c).isNull().cast("int")
    ).alias(c)
    for c in df_combine.columns
])

print("="*20)
print(f"number of null for each columns")
df_null.show()
print("="*20)

df_combine.show()


PySparkTypeError: [NOT_COLUMN] Argument `condition` should be a Column, got bool.

In [20]:
print(df_combine.select("STP").distinct().count())
print(df_combine.select("SLP").distinct().count())

959


753


In [5]:
"""
This code will stores all the csv file into shared folder

Running this code will overwrite files in shared folder
"""

current_path = os.getcwd()
root_path = current_path.removesuffix("/task1")
stored_path = os.path.join(root_path, "shared")


df_combine.write.csv(path= stored_path, mode= "overwrite", header = True)
print("="*20)
print(f"Successful store data")
print("="*20)


26/04/08 10:40:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Successful store data
